# Module 15: RealTime Chat Presence System Discord — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/chat_presence_platform.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import chat_presence_platform

classes = [n for n, o in inspect.getmembers(chat_presence_platform, inspect.isclass)
           if o.__module__ == 'chat_presence_platform']
functions = [n for n, o in inspect.getmembers(chat_presence_platform, inspect.isfunction)
             if o.__module__ == 'chat_presence_platform']

print('module   : chat_presence_platform')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(chat_presence_platform, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Cross gateway message routing

This is the module's own `test_cross_gateway_message_routing` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
import time

from chat_presence_platform import (
    DistributedChatPlatform,
    UserStatus,
    WebSocketGatewayNode,
)

platform = DistributedChatPlatform()
gw1 = WebSocketGatewayNode("gateway-us-east")
gw2 = WebSocketGatewayNode("gateway-eu-west")

platform.register_gateway(gw1)
platform.register_gateway(gw2)

# Alice connects to US gateway, Bob connects to EU gateway
platform.user_connect("alice", "gateway-us-east")
platform.user_connect("bob", "gateway-eu-west")

# Alice sends message to Bob
msg = platform.send_message(
    sender_id="alice",
    recipient_id="bob",
    conversation_id="conv-1",
    content="Hello from New York!",
)

# Message must be delivered to Bob's WebSocket inbox on gw2
assert len(gw2.connected_users["bob"]) == 1
assert gw2.connected_users["bob"][0].content == "Hello from New York!"
assert gw2.connected_users["bob"][0].message_id == msg.message_id

print('PASSED: test_cross_gateway_message_routing')

## 3. 🔮 Prediction — commit before you run

A user is connected to server A. Their friend on server B sends a message. Predict what infrastructure is required for delivery, and why a plain in-memory socket map is insufficient.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_offline_mailbox_queuing_and_flush_on_connect`, which tests exactly this property.


In [ ]:
platform = DistributedChatPlatform()
gw = WebSocketGatewayNode("gateway-1")
platform.register_gateway(gw)

# Alice sends message to Charlie who is currently OFFLINE
platform.send_message(
    sender_id="alice",
    recipient_id="charlie",
    conversation_id="conv-2",
    content="Read this when you wake up!",
)

# Message is queued in Charlie's offline mailbox
assert len(platform.offline_mailboxes["charlie"]) == 1

# Charlie connects to gateway-1 -> Pending offline messages must flush to his inbox!
platform.user_connect("charlie", "gateway-1")

assert len(platform.offline_mailboxes["charlie"]) == 0
assert len(gw.connected_users["charlie"]) == 1
assert gw.connected_users["charlie"][0].content == "Read this when you wake up!"

print('PASSED: test_offline_mailbox_queuing_and_flush_on_connect')

## 4. Measure it: Presence heartbeat and timeout

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_presence_heartbeat_and_timeout` and times it.


In [ ]:

_t0 = time.perf_counter()

platform = DistributedChatPlatform(presence_timeout=0.4)
gw = WebSocketGatewayNode("gw-main")
platform.register_gateway(gw)

platform.user_connect("david", "gw-main")
assert platform.presence.get_status("david") == UserStatus.ONLINE

# Wait 0.25s -> Transitions to AWAY
time.sleep(0.25)
assert platform.presence.get_status("david") == UserStatus.AWAY

# Send heartbeat ping -> Restored to ONLINE
platform.presence.heartbeat("david")
assert platform.presence.get_status("david") == UserStatus.ONLINE

# Wait 0.45s without heartbeat -> Transitions to OFFLINE
time.sleep(0.45)
assert platform.presence.get_status("david") == UserStatus.OFFLINE

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_presence_heartbeat_and_timeout')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(chat_presence_platform) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Presence is soft state with a TTL, not a database row.
2. Cross-server delivery needs a shared bus - an in-memory map cannot span nodes.
3. Fan-out on a shared channel is where the cost actually lands.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
